# BS5: khôi phục kết quả từ Dataset đã lưu
Attach **Dataset kết quả** có thư mục `continuous600`, rồi **Run All**.
CPU là đủ; tắt GPU và Internet. Nếu source snapshot/source giải nén không còn,
attach thêm **Dataset source gốc đã dùng lúc chạy**, không dùng bản source mới.

Notebook kiểm tra receipts/budget và toàn bộ raw slots, sửa bộ đọc trên bản sao,
kiểm tra lại ZIP rồi xuất báo cáo. **Không train/eval thêm** và không sửa Dataset.
Weights được giữ đúng theo SHA256 đã ghi. Thiếu training export vẫn xuất được
phân tích đã audit; thiếu weights sẽ báo đúng file, không thay checkpoint khác.
Nếu trajectory thiếu/hỏng hoặc chưa hoàn tất, notebook dừng và giữ bằng chứng.
Kaggle có thể giải nén ZIP/GZIP: khi chỉ còn JSON, báo cáo giữ nguyên các kiểm tra
nội dung nhưng ghi `CONTENT_ONLY`, `accepted=False` vì hash container gốc không
còn kiểm chứng được. Không coi đó là integrity PASS và không sửa protocol.


In [ ]:
from pathlib import Path
# Some imports are shared with the following notebook cells.
import base64  # noqa: F401
import hashlib  # noqa: F401
import json
import os
import subprocess  # noqa: F401
import sys
import zipfile  # noqa: F401
import zlib  # noqa: F401
from IPython.display import FileLink, Markdown, display  # noqa: F401

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

# Leave empty for automatic discovery. Set an exact continuous600 folder only
# when multiple result Datasets are attached. If source bytes are missing,
# attach the ORIGINAL source Dataset used for the saved run.
EVIDENCE_OVERRIDE = ""
required = ("checkpoint_selection.json", "protocol.json", "run_registry.json",
            "budget_ledger.json", "controls.json", "source_identity.json", "source_snapshot.zip")
if EVIDENCE_OVERRIDE:
    candidates = [Path(EVIDENCE_OVERRIDE).absolute()]
else:
    # Kaggle can expose Dataset aliases through directory links. Path.rglob on
    # Python 3.13 skips those links by default. Walk them, with cycle protection.
    fast = set()
    for pattern in ("continuous600", "*/continuous600", "*/*/continuous600",
                    "datasets/*/*/continuous600", "datasets/*/*/*/continuous600"):
        fast.update(p for p in INPUT.glob(pattern) if p.is_dir())
    found = {p.resolve(): p for p in sorted(fast)}
    seen, walk_errors = set(), []
    scan = os.walk(INPUT, followlinks=True, onerror=lambda exc: walk_errors.append(str(exc))) if not found else ()
    for base, directories, filenames in scan:
        folder = Path(base)
        real = folder.resolve()
        if real in seen:
            directories[:] = []
            continue
        seen.add(real)
        if folder.name == "continuous600" or "checkpoint_selection.json" in filenames:
            found.setdefault(real, folder)
        directories[:] = [n for n in directories if n not in
                          ("src", "scripts", "tests", "configs", "runs", "parents", "exports")
                          and not n.startswith("repro_")]
    candidates = sorted(found.values())
    if walk_errors:
        print("Input traversal warnings:", walk_errors)
diagnostic = []
for folder in candidates:
    missing = [name for name in required if not (folder / name).is_file()]
    diagnostic.append(dict(folder=str(folder), resolved=str(folder.resolve()), missing=missing))
    print("Found evaluation folder:", folder)
    print("Missing required files:", missing or "NONE")
if not candidates:
    mounted = [str(p) for p in INPUT.iterdir()] if INPUT.is_dir() else []
    print("Input entries:", mounted)
    print("No continuous600 folder or checkpoint_selection.json found. Check Add Input.")
    diagnostic.append(dict(input_entries=mounted))
diagnostic_path = WORK / "bs5_input_diagnostic.json"
diagnostic_path.write_text(json.dumps(diagnostic, indent=2), encoding="utf-8")
assert len(candidates) == 1, (
    "Attach ONE saved result Dataset containing continuous600, or set "
    f"EVIDENCE_OVERRIDE to its exact folder. See diagnostics above: {candidates}"
)
EVIDENCE = candidates[0]
assert EVIDENCE.is_relative_to(INPUT.absolute()), "Select the saved Dataset under /kaggle/input"
assert EVIDENCE.name == "continuous600", "Keep the continuous600 folder structure"
missing = [name for name in required if not (EVIDENCE / name).is_file()]
assert not [name for name in missing if name != "source_snapshot.zip"], (
    f"Missing saved metadata in {EVIDENCE}: {missing}. Do not train again to diagnose this."
)
protocol = json.loads((EVIDENCE / "protocol.json").read_text())
missing_references = [name for name in protocol.get("evidence_files", {})
                      if not (EVIDENCE / name).is_file()]
if missing_references:
    print(f"Original reference containers absent: {len(missing_references)}. Reader will check extracted JSON/source.")
    print("Original compressed hashes remain UNVERIFIED if only extracted content survives.")
    json_refs = [name for name in missing_references if name.endswith(".json.gz")]
    unavailable = []
    for name in json_refs:
        original = EVIDENCE / name
        plain = original.with_suffix("")
        options = (plain, original / plain.name, plain / plain.name,
                   original.parent / original.name.removesuffix(".json.gz"))
        if not any(path.is_file() for path in options):
            unavailable.append(name)
    print(f"Extracted reference JSON found: {len(json_refs) - len(unavailable)}/{len(json_refs)}")
    if unavailable:
        print("Missing JSON examples:", unavailable[:5])

def restore_eval_source(evidence, search_root, work):
    evidence, search_root, work = map(Path, (evidence, search_root, work))
    identity = json.loads((evidence / "source_identity.json").read_text(encoding="utf-8"))
    protocol = json.loads((evidence / "protocol.json").read_text(encoding="utf-8"))
    files = identity["files"]
    digest = hashlib.sha256(
        json.dumps(files, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()
    ).hexdigest()
    assert digest == identity["sha256"] == protocol["eval_source_sha256"], (
        "source identity mismatch"
    )
    assert files and all(
        not Path(rel).is_absolute() and ".." not in Path(rel).parts for rel in files
    )

    def matches(read):
        try:
            return all(
                hashlib.sha256(read(rel)).hexdigest() == expected for rel, expected in files.items()
            )
        except (OSError, KeyError, zipfile.BadZipFile):
            return False

    destination = work / ("bs5_verified_source_" + digest[:16]) / "source_snapshot.zip"
    if destination.is_file():
        with zipfile.ZipFile(destination) as archive:
            assert matches(lambda rel: archive.read("uav_thesis/" + rel)), "cached source mismatch"
        return destination
    seen, trees, archives = set(), [], []
    packets, origin = None, None
    anchor = "AGENTS.md" if "AGENTS.md" in files else next(iter(files))
    for base, directories, names in os.walk(search_root, followlinks=True):
        folder = Path(base)
        resolved = folder.resolve()
        if resolved in seen:
            directories[:] = []
            continue
        seen.add(resolved)
        if Path(anchor).name in names:
            candidate = folder
            for _ in Path(anchor).parts[:-1]:
                candidate = candidate.parent
            trees.append(candidate)
            if matches(lambda rel: (candidate / rel).read_bytes()):
                packets = {rel: (candidate / rel).read_bytes() for rel in files}
                origin = dict(kind="verified_extracted_source_tree", path=str(candidate))
                break
        archives.extend(
            folder / n
            for n in names
            if n.endswith(".zip") and ("source" in n.lower() or "snapshot" in n.lower())
        )
        directories[:] = [
            name
            for name in directories
            if name not in ("trajectories", "control_raw", "runs", "parents", "exports")
            and not name.startswith("repro_")
        ]
    for tree in sorted(set(trees)):
        if matches(lambda rel: (tree / rel).read_bytes()):
            packets = {rel: (tree / rel).read_bytes() for rel in files}
            origin = dict(kind="verified_extracted_source_tree", path=str(tree))
            break
    if packets is None:
        for path in sorted(archives):
            try:
                with zipfile.ZipFile(path) as archive:
                    if matches(lambda rel: archive.read("uav_thesis/" + rel)):
                        packets = {rel: archive.read("uav_thesis/" + rel) for rel in files}
                        origin = dict(kind="verified_matching_source_archive", path=str(path))
                        break
            except (OSError, zipfile.BadZipFile):
                continue
    if packets is None:
        raise FileNotFoundError(
            "Missing source_snapshot.zip and no complete source tree/archive matches the frozen "
            f"source identity {digest}. Attach the ORIGINAL source Dataset used for this run; "
            f"do not use a newer source ZIP. Checked trees: {[str(p) for p in trees]}; "
            f"archives: {[str(p) for p in archives]}"
        )
    # Recheck every byte immediately before creating the derived container.
    assert all(hashlib.sha256(packets[rel]).hexdigest() == value for rel, value in files.items())
    destination.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(destination, "x", compression=zipfile.ZIP_DEFLATED) as archive:
        for rel, data in packets.items():
            archive.writestr("uav_thesis/" + rel, data)
    with zipfile.ZipFile(destination) as archive:
        assert archive.testzip() is None and matches(lambda rel: archive.read("uav_thesis/" + rel))
    report = dict(
        origin=origin,
        frozen_source_sha256=digest,
        source_files_verified=len(files),
        restored_file=str(destination),
        container_sha256=hashlib.sha256(destination.read_bytes()).hexdigest(),
        original_container_missing=True,
        original_container_byte_identity_claimed=False,
        source_bytes_unchanged=True,
        added_training_steps=0,
        added_environment_steps=0,
    )
    (destination.parent / "snapshot_recovery.json").write_text(
        json.dumps(report, indent=2), encoding="utf-8"
    )
    print(
        f"Restored eval source ZIP: {len(files)} files match frozen SHA256; original Dataset unchanged."
    )
    return destination

SOURCE_SNAPSHOT = EVIDENCE / "source_snapshot.zip"
if not SOURCE_SNAPSHOT.is_file():
    SOURCE_SNAPSHOT = restore_eval_source(EVIDENCE, INPUT, WORK)
FLOW = EVIDENCE.parent
registry = json.loads((EVIDENCE / "run_registry.json").read_text())
complete = sum(item["status"] == "complete" for item in registry["trajectories"].values())
print("Input:", FLOW)
print(f"Recorded trajectories complete: {complete}/40. Original status: {registry['status']}")
print("OFFLINE RECOVERY: CPU only; no training, model loading, rollouts or Internet.")


In [ ]:
IDENTITY = json.loads((EVIDENCE / "source_identity.json").read_text())
PROTOCOL = json.loads((EVIDENCE / "protocol.json").read_text())
identity_hash = hashlib.sha256(json.dumps(
    IDENTITY["files"], sort_keys=True, separators=(",", ":"), allow_nan=False
).encode()).hexdigest()
assert IDENTITY["sha256"] == identity_hash == PROTOCOL["eval_source_sha256"]
TOOLS = WORK / "bs5_export_repair_tools"
TOOLS.mkdir(exist_ok=True)
NAMES = (
    "analyze_bs5_continuous.py", "bs5_continuous_common.py", "bs5_bundle.py",
    "analyze_bs5_curriculum_bundle.py", "analyze_bs5_curriculum_extend.py",
    "bs5_curve_plot.py",
)
PATCHES = {
    "analyze_bs5_curriculum_extend.py": [
        ("def gateway_auditor(config):",
         "def gateway_auditor(config, *, require_exclusion_flag=True):"),
        ('        assert row["failed_bs_gateway_excluded"] is True',
         '        if require_exclusion_flag or "failed_bs_gateway_excluded" in row:\n'
         '            assert row["failed_bs_gateway_excluded"] is True'),
    ],
    "analyze_bs5_continuous.py": [
        ("checker = gateway_auditor(config)",
         "checker = gateway_auditor(config, require_exclusion_flag=False)"),
    ],
}
with zipfile.ZipFile(SOURCE_SNAPSHOT) as frozen:
    for name in NAMES:
        relative = "scripts/analysis/" + name
        data = frozen.read("uav_thesis/" + relative)
        assert hashlib.sha256(data).hexdigest() == IDENTITY["files"][relative], relative
        code = data.decode("utf-8")
        for old, new in PATCHES.get(name, []):
            if new not in code:
                assert code.count(old) == 1, f"Unexpected source; stop: {name}"
                code = code.replace(old, new, 1)
        compile(code, name, "exec")
        (TOOLS / name).write_text(code, encoding="utf-8")

PAYLOAD = (
    "eNqtXG1z3LiR/j6/AsV8yIxDUVrH2UrJxarzruWKbx3JZXmT1E6pWBSJmeGKQ/IIUvLYp/9+3Y0XAiQ4kpPTB1szBBoN9NvTjaaC"
    "ILjabMqi4ox/aeq2Yy3P6nveHli66XjL0rJkr85YVlddUfV1L348O2Ndm/7Os65uCy7g0b4pecfzaLG4rNnnus12IePVfdHW1Z5X"
    "XYjji6qotiHLdjy7a+qi6lhZpzl9V7esrcuy7jtW7JEHEbGrttgWVVqKRcv3MJndHjp+UuRArsjSMmJvWM7b4p7njN/j1xkHRpoD"
    "sd/mgnU7zjZpUeIAuTHRpV0vwgWOqETX9llnmC/qitVVeWCbtt6zvG/T25IjLV40nQhZWuWsECzt8wI2yooq5w2vkJvyEC2CIFgs"
    "aGaSbPqub3mSqK3AzKqGhWEBsVjo79ptk7aCyzl4tPxLVxa3eo76Zp9W6Za3etYuFTsYpD/+LupK/14LSapJu51F5yN81EPEru+K"
    "0nzqb5u2zrgQ5puD+bXj+2YDJ2c+F3vz+9dCPlr87eLTBYtpiSXsGr5LklXUclGX93y5imCDcDqLYgPn3i5x9IrBScDR4VIRcnq+"
    "YPCjP0VFJXjbLc/CYcZKHSscRHn4ypNb8Zdk0MThiOlpiIIGreR5cscPILT7tCzytOMJKAGeI5FCErd9lYN49fSu3hdZgucJ+plW"
    "dYUaluBxh4w2Jn9teZrTqMXi89XVh2vY/ZJ2EPjZi5pDEMoB7gP4db+vq9FzyZT1pUO1b9si68t+/+xxoECgoWMm+vaeJ01Zd9YD"
    "ZfEj9hNpNnIcCGLxXyO1zPmGFcBKUtbZ3RJFuJISBXN4wxreikIAC2jo2R179/7DBZpQVbf7tHwNp7npBZdGlzLkm4yJ7XiZs6tr"
    "mhShYSFF0CLUHVIT/gXIiqVaC38OBcwxn1oOBljRxz+wX9LtFiT9Nu1SwTu2r/sKjB5Uk6R5gqtH7OPV9ft/sQ2xCR6nZg91eyeA"
    "NWAMh7F3byOi91B0O8lEDda/DNo/3QbIWy2iKt1zFscsqLqA8RJ2FrS3wYqlArWZp/uB3a49DB/U7kYU3Oc0RurqXtxnbbeYPJbf"
    "R7gFcKlLuWSEylvVy1Won3/4Jbn86cPPv4Tsh5VDAzmeXXSTgWSma9LXER3bdEH58MPVz78kF/9i/2t/vvxpWJt/yXjTgbwv2hbC"
    "AJwWfOMyAqEDTvMTCA7cEA1bbgK09LbA4II6JcCzlaztK4wxp8gQz8/ZN5TUI8iA7B7ozkvA1aANBp5yNETtUHB+tzxbfbf8ni+g"
    "Xy+fL6DniuDXSzRgNFi0YQinPF+2dd2hUyshON1zZU54ZOjX8CE7HZ4Ojp2GpQJdtTSFQiR6WNLVNHMYjWwEfSXSDafRIBU9+FGa"
    "tjRXeqhYLKp7kGvdHoiWYgwo9mUHrH17pM8b0JZbsOmQ5UWroYh01wJjDAjkIS3v1C43CC8eAOXcifgdmDi3vAdS0YEMfx/OHdcg"
    "qQI9axX2J7mMK44GaBCpU5rjPFPHRS4Mz0sc9sjLckW4Ar/ecsAIEPWakAUw4Pe+yhAyBCEr0/1tnp4zyTbKdTlVgwDJWVAIlV+d"
    "r/7xb8u3EakCc3tBV6w1HSyONNqj7yqOcHTCypuzfXrAxW8LZJRX5FvBGFgBLhnhHXrcfwK0qh+E6wFI9mvStrGqraJUJE0tii/L"
    "1Q2wbeK1DEi2ikkyxg4UaE00yHO1bQtBBjBwPER9bRO460QPiPBJIJeRKMM75baHR53CIfacbAdYkOcIXXjWI7SMx7BlKf9zDE9N"
    "QyEME2NQ0qxbIlqOX746OwMghVIAaB3/8PLsDDQn4EC7JzBKWqcPIbBpg/tQS64D0EnAgp0IblZI/9VfxwO/pesAwFZwQ1qVokpN"
    "5z56JoODHfQYiEhwDnRQrwxfxj5gBJgETITNJYAqGpH0TYOYBSJ6Htw4yj3DBo3BU+gr8PCF2MGhbUGVXw8pTwV6CckIb8r0AG6k"
    "cw4GAMRSy30d2CmQOh4HfrqeErfLZ3eJTJM5zpGPUG4caDpEp3qm5C2Uis0taDyoOhz/eQ1GrVzCOHaooetAmRAcw2Lk8kbmiGyM"
    "ZyVil778y49KPsZW5UEoa4Wsg1IMsexQ8Tvl6JW1Wu5VPjYYEaMPUMT8kb5Cb5O2XbFJsw58pBytHCVyVbcpGb0ig3gvEf1mA94l"
    "iICJqNs3wWqyqJ7pWxcyXd62fdOZRBQX1jPU2nhCEFgguqFvPZ8cm3bZoRQF5lBACARBxxBBTrpH5XgcUKrK06LfiuYd/L80K4L+"
    "f4GwkmJAhIc/voo/tz0noJpC4g6u9XwSKZ5Y1vX+ikr00MLzaaxCSuHkW1rFE0T2DXhtkXSHhsdTWvhjNvr+Y3L9+erTxVsMURQr"
    "pOSQbZDe9ivsmoQIiJAAuj3z7cW7D28+X7ydLLEKZ6Kos01MWJ1xwfXf3oBSX//692tpiyHl61He7xuxlOIOqYxQdfHLVTja6lHW"
    "Vs+Qsl+e2hkp1pFvoAAwBDD0ZV3x8UD0eHowSqgE7QZ5oxXjI7mNFaD7b5P9Pro2BXEbCSjzdcIyBS2UV4ynqAaETHqFeLAE8wSh"
    "goiViaKDW67gv0QUX7lGubrelOTAM3r2JaZ0gJMfgDAvOYErpbdgpSVYnh6AblRPP8Vnf/7xThpp44R2mgZjm7bu6qwu7aiOW0Vr"
    "Nkutg30NrOgg1NV3AH/iUbFh0CA6EheUqVXixtVHrCQAfIqV35iyB2BFBArHyQiBExSzDhQkph/DkZppUZGrJs8o6UpXRqQ3xpN/"
    "o409Suoyca82tYuI5FDHh+IgCFVV2ohd3clgJY8Iwz/CDDlCe24db/8s4xhBfpMYUImOzhO+M0oNIb2t7S9w5+i/GLnoEfUjkQ/3"
    "H9LMdYAP7ZhnVykAxCNjy5UvmUVbvay7d4hdZEb7EnzT3wshMEDVqvRptFgbbCArYxQSnhNpJZdOeB0nb675DJPIxmxYhYe8Hgwj"
    "gCSICKDWIO6m/M06CPMQlwBFQXe1DKRwymK765JgHDdswc2wPkuY5Jts+rKc0NWiNxxqL2it55iqgXFyhGaClFESozJWNyiTwWPw"
    "0NQBfBRphNqP14HTAL/z3qdVsQGPDVuhUILlc2HcM1rYcuKFV27WR07I2LoBlePzkhkWrEPjEaipnEsqvC+11cyt9VgyYjWfVHMi"
    "S2u2qmxHcoy7J1OCWEU7/iUvthxj0FHqtldbBhsIq7tkKBjpeKwe+FKz86N6r1XeFzECO2X2TVb+z9iN+myP13QlM+OV3Kdmvxqg"
    "SWQ2lez/k28yLkkHVlpw6phUbCduVEzWMxSQ9wTkkL0ISapDHfmTzMwAOYNnViMBSJN7Yg87DBGmuFum2Z0YOFS3PvJAdCFZqyss"
    "orQDXAYqQ9p3tYreTnVQR5fnIYqFVdOcHOSkugkyIUYwNyM+PPKYMKLCHG3mHoJReluURYdJyxQ16IQvMdzLQ5E1MDhrKdXhMS0S"
    "r1GYwKmsJYKirm5GqMCtRtrCDr+HDcw8XDwzx9GNO0wrQvLAMZbgjUoPQBqhggdzjTB8UWVln3ORjKkc4Wcy9CmO4m/eHOfcczxj"
    "x0tuNt5P/G7onaVw8n7kCP2Dyzqj6k98JDhNZ66mBUeTGBKA8py5Tg6dqRN0iXUEAsoyrWykK0OSxs1ty/pWx/cXAFkRAqAHy4tW"
    "l0OISpQ2eCO7PIrj5Xgw32JTYNVNiZfMKmRrN3yEQ7w8ssHBDADN51S4wx2tx8USfyyVEJzKN1j+wY0MtoYJId5wADXrZKxl9FlI"
    "b34zcfZy/rlPo/VxTUQ7r5+0g9ifqXv017tf/0yjglqZPZF9Rvs5ggWeS5/VrHzH9KRGrxa+yrWDnHR49R7pPAL3bsRbwDehzWqQ"
    "IGwK8tqnXbYz91kzrMt8GCvgwdgXnVJ2+Hj6je44JE9/dATzx5sVIRKLugQjkioBfLnz9dmgZtqO1hrMTHXn+3Xi2Sog6r7NOBUN"
    "Bt5Grl7elQOS1/WCsQOyQ2jUN6gvwwYmPt04jsWT8UE9CBfPCFmarMX8fIxyJqCOHQ96UzgWOptW4ExVwOXeDbKhT3XfNb06uBfy"
    "PwPlcLFYYSgJDAB5JlIyiU7oY8yT5GOqOmJnQpvSprSTbNW1XLhQ/tR4cXUxpz+PLyIpiRqaEGznby6nVuHMCKen6XQYP0E2ukFC"
    "3iy7Gu4elv6xD814VufUnE/uQO8h+r50px053PlHtkE4quBsdsC6amOI1N0deLkOjzClRU1pPF184QV/BGnaRtoGqKMUctOCNxxO"
    "3TSs6Wubc+k0MZCbaz3Tu/Wa/Xbx6crg91PrCozukSJA/5uyFztCf5bNjPd8RBVD+USOm7krx4AsB4xvzA3B0LTiHPSa+x4S/luO"
    "H0UBqYKpDembXumwzddurugouL5uF13dYqglSdklMUANPhna1Y5JKRkfLM3qsORosqy3DxhlHPyCvJYnA5tucYOpdVOjWDSFwcU0"
    "6nkZVoLyPRsLh/ZNTYWde9nr2ZMeZ1d6raNwN6YHe4pY+BNPir/DBPLSdl1RTfAzN6o/rwNn2+O1B6Le8pP3yLzVKA2MQJEx5cOq"
    "DJVSx7uYwuTvqP0EfXqfdDsuCqGKJ+W0DCR/p04Wy2+PtXwKTTZt/ZVX7knFU7GFE3BI7iuZ3E94z24UFZTcBi+YKJzgST09g7Hs"
    "YdQwycoUHGauI+cosxo6GDytDpr2yvEeibmGHFpw3JFdXZfea0rqBpWlL/dWgTolH0eXCMq5nQ7+TrNmXx1sTLFs2vIHiO++wBbQ"
    "2bsFu9KtBq8Ds1PtPofq7tQiR8dyhCx1YIKSJnQ++gIDfp2f4gRPOcP5am5mBCAW02Gl35bSBSvpJBzNP05mPjhDUJQNR3RjPx/D"
    "Z1Mg8sDDdlW2S+SGbyfuSTeDqREz8XO4ZR8u2bWSG1DwGkBBXQuIKOzy4p/s5EQpnW7kOgQWuo32d1hPkD3Lsg40lBYchXXjLjW7"
    "ybkr1XH6AfMMVuz3fUc95Fvwsae/vf8IKSTgKew4hxAHG285sCLu2C3iLBGxN2XJ9Jz/vr66VOT2sEHISlKMwlndUNVC1KqLtvhK"
    "QOZU6x/Spr6eXVoBE9gCr5VYRAvLZcsCP7bJuTru1CZJhEMH4FNdX3bHl9X8Kw5VtmvBriSvpCIhSVNkBXqyTZEZNLNwkzvEmAgF"
    "5Ez7Gk7zbTUwhvO9jfauxqHBQjfqhkdOcq6XPKjmdHzP5LI6zfFN+5h2eXoPp/xLAyuPrNnqvoxEA6naMjgNsC10/cPNTAmAdqcv"
    "b+JpD8Ixfue5G1NZeAiobn+PGYUSziX1nWVVilm50/k2DZffSdMuma+IqJPSoy+r2Z5EN9lyOpDPPf0qYnBz8iWKCF80ITw7u+xx"
    "RTufJehPorRwvODac+U17VCEM5VFA1n/NV9+uvh49elz8o/30T63D9zgJgtqOq2gg/hmbpNyeVU5q1eBr7kUJ/0buvSUaFQiNrzn"
    "caSXU11g6C9D9dpQPHTPrabkFmMjlxccid61In2kwQNQDAyJrZSNlfUW+0lrbPBgJd+m2UG90ZTcimSbdvwhPUBYpkpRbiLFa3mz"
    "zSBW9yC3LIXAU3QFtUVbby0FM2hTbXfoRNQthDPjR0gqfgJEzVCR1zyjwaPyhgOyYvp3NCInJzq8eCY7ReMz3zBzw+QdUzddsS++"
    "AuaWRcHJgDZ9SNr6QSBYo1rcGIb7D+nYaB+oi0deZFQCJdEk1gtteLeDsO6V9Y7egcFRUOM6RJlXfzVvt+lOT5MgvGZYyWO1Kq9I"
    "NVLvVvD/6QE45ZbWOAVGg0BsrKS/DKwqTvAGqWJ2L1/UA8UVJV4CYey1at+Q1kIKksKZRZFbolE5S0Ov1OnM7kjtS7Y3gr+aHyP7"
    "Ab89um8aqPeqdLOr2kyozyLRx6Z804sXiifV2Y0vS1XorChHQZoSdOsHtFmRYHzBc9Vd7kGaYUCCr25Gzb7jx/je5JGNyxM1XMz7"
    "QF9gkKvJOWPPamR9OoofQ6Byw4rbptUc7eE/Z9ZVl3yXRN4A2nd/Mt48DnQnpXv39tVoouxbchjQj0ZMTAodxJAebDE1fjnCYtA8"
    "erQsxeFX5dFW9uxJnldDXxuc39AKPRiaU+fTQcc0MFG7FWEF3TZt2+MnboqnmC9glpJu7bdqqVCEj8DbVHlaSqBMptH6bfMP7B+o"
    "UgeaJTs3DFnZudELvZ66oQHieVncnsjXATX1ocSvX0mNPuuO0rc6j8MsE/BjHOg14ADkximWt7Eqy2IZC8lYL+LBJ10sxN+fKI4R"
    "UZPKzlbFdAVL2SR2/tNCdvI4rIT9mGioMVu7rv0gIvlqB/pq1+0HJz8F40DQThY5nX8rdfX07PGYQCfQR1Y2/t6a61xKzzur80lX"
    "KxyKvqOGtWnmiZl5YlUqJmdpTVOu+mTAbgPCNW9ARwACl2quekHdUmZzteZ9sWeIWIkZaNeHlZH9rE1MXb1B9LWaO1XfKTnsUaOT"
    "uvXz25kqGGFJYNwK9Ixeq+kt1PRKNFH9r4OrMbDJuTkdEkM7wEwouXyONrE+RhtvlyfkxvOHq53T53rEG23/rnGPKc+L+cbKceZH"
    "TTj1pQc2Ke+AMZVp4LZJTJ+6PdeTCi/WDMYInAWTGydVXcoRR2ochzxK8urVen+cQk/UchIzgVxXNhJzCBWpzD512JTXW27mJFcb"
    "vI0GR/EULYULuwwwAWDxgJTsW/iOb1tUN5MVDThO5UVwRB/fXF/bPu8/vIidv/u3LWAdzI6zNzvTduBS8g9yz2ymhc8lNDfMJvWs"
    "a3AAYp1yzwligtH9yDMyvaeyvP88gZ1S6CtlGyN2vy+zO57oAnRtBM6KPdfl7ERfpzvJmQ365R1JKA1q1B0C36gmAHyJdGneCW8F"
    "vdeq/2xJ9Kbd9njqH+nJEnQwawtKfuIkyessSVbWzAhEkaRqCkZlHY7Aduhtp48UGnSKaQW4mfkaiPx7s49AiRCLJpRCk2hAe3oN"
    "GXzEFhY4QsmeSEmemAvrQWoDp+arHS+bONCtPLriZrhhXY06LP9ABnU62+V8uvhmeK2Af5XmFrufAlvkT7Grze6Erp8sO9/VBRhd"
    "PG6Mts13k4IPdLp+hu3g6HN2x3lj/laO1XBBb33Lvfi6tOnvcqgLUWc3sAVBb3DQpug/3Jbw94dYL9qN6itWj5OL2Lci8vfxmMe+"
    "hp6pN6OhRzp7Zrt7aOLTLT5PRRek8pwQ43mzUb+LOH0BbIF/xCdJMClNErpYSBL0D0mibhSks1j8H1PzinE="
)

helper = zlib.decompress(base64.b64decode(PAYLOAD))
assert hashlib.sha256(helper).hexdigest() == "cd496d44f522cc3c7ad7aa8ab1034c40c7bae7403065bdb0838c1280435e34a4"
TOOL = TOOLS / "recover_bs5_continuous_export.py"
compile(helper, str(TOOL), "exec")
TOOL.write_bytes(helper)
print("Tools ready: frozen-source hashes checked; only reader copies patched.")

ANALYZER_PAYLOAD = (
    "eNrtfV1vJMmR2Dt/RaoEQ927zWZzPla7HLWA0Qx3RWmWHJOcFXR9RKFYnd1dYldVqz74MTwCd/CTYdzDwgaEgx98K0Ew5IPgO5+f"
    "dh7uYYT7H+Nf4ojI76qsZs/u2r4DvMByqqsyozIjI+MrI6KCIDipomwaLfOMsyK6YnGeVUlW53XJpkk0z/KySmIW1dOkGrLDnJ3m"
    "RbwYsMM6fXkzYDy7ZHnBrngyX1TlMAiCra1ZkacsDGd1VRc8DFmSrvKiYlGW5VVUJXlWbm2pe8V8FRUlV78XUblYJufq56/KPFPX"
    "aVQtBOgVXEEjBfclPpDXJb4AB1yqO6+T1SxZ8q2tqrjZ22LwH8E4Lx+H53U2XXIFpkcP8b+oytMkDvHlA30zjrI8S+JoGeIYzX0E"
    "3ri1zKNpWBVRzM29gsM9F+JltEymUcXDAv6U5v5VkcDNuLwUt/ruoM3ywGWa5ll7/J8fPd9/cWIAHu+fHj89ONx/Hn6xf3xycHRo"
    "Hp082z98enxwZLX+xcHh86NfWDfO6/ImLPgyurEGOed5ygGlYbyMyjKZJbywcLXg8QWNFJBQWffzFFbbwsoymfH4Jl5at1YFnyXX"
    "YatpyZc8rkKJNCCi8JyXVdnCUZRFy5vXPCRc1UWRxPWyThtrzVdJmU95iFNAWrmvN7+ueDZVveewXlfRTUh7Ii8G7JIXyewmTKMM"
    "5lNWjQWri0serpZ5pfrj9Ra/jvmqYgd0a78o8sKizjIukhVsJxpOmZTDf5Xk6p3F/6ff9fTbQtq3I+hNwX1TCvevsZ/kt7amfMb4"
    "9QoQwafhBb8pe31B9QUHWZGx21lwm0Upv9u5Le8CNgPJgj9ZkkmioFsl/tYrfyfhltGM94o8rwYAbSnhoqhgY0b32Q49GBa8zJeX"
    "vCeQDqvPcYDQcJiURCdVAqOvcupkWveBOUwZiDDdtrxJl0l20evTG+15YAsYFtEW279MpjyL+WkRZSViQwwNROUx7DLAB1EZn5Lk"
    "jZKMFyUiaZnESbW8ecIyDthnACtJWbXgScHy5ZSd31ScxCUv4XWlFL0IGJERhkmWVGHYA7KbwfAILR8MWLRc5lehfuX402hZ8gGw"
    "gPgCfh2CDiARJ0l2NiTMjUnIEkKa+NMNG6ChT+OO21y8ElrJC1jX2zu3SZ0R2SXUDB7q2RU8zoupnNsynyOfg7nFVY3/KgIbsIsk"
    "g78ffEBDaE7MQJ9IEGfwmmkSV4YTCfZID0Nc07F6mdMiL5J5AvQflovoweOPxnoA/lZ6mYkPh2oUci2cPjiBMc3CuR1dRskyOgdO"
    "ToMSEx/apKuXrj+MynCVl8k1kGkHEDlsLRp6AmCfJTOJVKR2fNzrMw6DZEgoLjSJZHOzb5aL2ClvrZdCU5Ps1H1YDX2Ju95QzXDO"
    "q56EY6hQIRj6ES/QONDvNG1hZqq5mdueix/BGPQQkpKmjUMxmFJA+mw8tghvFgCBFjXId2y6x27lAO4C5xWSWygg9ui8WwoZkIQ0"
    "BG5dXiWwKYMhyunh/HXQmMAKOEYGyNATxeZhWc9APvWCoO9S54oUc+SVzn0NaNC6rfG9I1oMkVUP/L3vaaOHiDIzQ06t72AHIO00"
    "v+Rq6GbCLiR3RiCIS4QFHAYsgh5u7CGKLBI7KxIkKxQkauaA9JWhhb4LDB8KeHutwUtCgcXqGcJYEUVYv0Xvyeisb14tb4L8aCOd"
    "JGwQpefJnGwxQwU/Ozk67CSpNh5oiLSLARVmGO11im5QE4RGWvlTjKDVVmwtwYW9I/fySXc0/md+3qn+CzQWaHxBB3zN2bCRYm+u"
    "RtyT0+0PNkCf3Kdi3EZHjhJghZ/CEh/m1ac56GOkxvdmwecJqJTZ3OwQ3LhmAQsul4H0QFhK1e4usLgmaeSb8kyp59C6KHbb6tTf"
    "akzIaP49BEAMn5Qb2nQWh8G9Jji/oQ3qYYYrNUTBoGnY1vBSiZAxmxjCw22gxzhN5qBX4qaw+TxYFmnZZMwwyJ6fIrsUDk0Yhn/i"
    "7gxAVecFqmblDnQBjSmbh2VeFzHIxCxalQvQ/14nq8ALCDXBLjlj+IjLoNrMQ5pDnLHvs2egwTPUFaukukF5I1eSRXMYHGAHFcEb"
    "YirsnMNUhw447eBo4aVFEQLb7uCkQdqi5zZMuZrDaLUCGun5hKts4vbt2jBbbb6ndlAZXcL8udSgWS/jQJCgDeudlQERuayxDxtK"
    "vr7BGy1qFavc3F0K96Qqz4CazqP4ornVvmM1I17kCRAgbg3V1GwRvuxQBe6F0RT2IFKDOroMAXtgpQUDW3rbD86aW02hwUO64q2K"
    "DJrIUB0tSYocpHPkLmMAbj0ldwOyBNlq735yhwHrnl0Y144LwJByDw7/LFkhYfZ0bzD3ShYV8QI0an9/2gpRFS9o/Cj+O5vhf9K1"
    "ORTiqCdBD5Gh9qwV2AnYh2Sn9ocLfi02ao+UCXG99h2IOehrs1NF05MA5wkLrJhqJ6D2EyJDB6/TpOhC64YoMaqRWWphoG8w1289"
    "T98cy451Vizaxy17RyfEwwbs5/xGXimS+kk0lVTlwZTBEpl9zV0nH3sJHAXPfVxlY0XtXmVNOhPFCnU36bLOunsEUtIac1hy7WDN"
    "W0QXoh214MaAPi3qNSME3f81z7R8F0qhoRpxIzgbbEgvQoXa6mI+SMRqkdpLIyThUzIcQAVUeqMYG7l2ShShRAW2qt9/D+WTOpMM"
    "FVBbWqcFVXjQSLcr8qtSutGSbMqvSRZWynVlC0NsaRi3tIKWPOtRP9rG+Os2mQSoKwZntG0T2qzY4g7snmBao6MLERYv6uxCCGxq"
    "B7tXN92zJaoejDLJG2avVIWNOxBBqTH0fTa+cuhJFwdqd5blRpoxTEaA0XTCfHqLNPt3lMJLk2K3CKOljNi/gGQi8oW5CvkaySl7"
    "aGQMRTd7roPGkA04Oe+kRLUyAs2qh+AGZCmL+ePK4T175su8shUEXP+h8BuLprYHFB9KsjIHB4K0rkqiKfIEsDLmWVQk+QDZ7CyZ"
    "D9gCts3rPBt/NBoNcBjzaoHXkujkyCeFGI8kqgIpBSGfEdWhvQ94mfPergIBQnW3jwQnV2bHEB7BsYGP2I9Urx+N1XioAero+nS0"
    "rPgqFLyVTXPYsciYkyxe1qDGgzAHGPMovmHKtGApryJattkymg8lwJO6uEwu8fFPTtjBc9Q5uFb8Sct/QsBwH6CehS6NOFpFMdoI"
    "/Ne1OFCFwSTLpQQJsonG8JOTx4QdYTYI63UHiGGaX7GMV1d5cTFkz3Mad5pPwXxTCs9ULO6WPgwBlXvcPBXoqRUrYBhJwUN6b4lH"
    "GzhB4c8URCEPWW4QSPvgpWcdsCDESTAvkmlYJq+5zY/Vw5gvl/QwTH2PoxhUxTIsommCZ0zQuPQ1Q9V0EdXLzoYVEAjvib8ry2mj"
    "AZTh9Q0OoW+f4URFCkTNyXtJVnQJZFb1ghDvSf7NkVOXwqet2d0UOC2RMc/qlOMZG20V22u9JE88tERibu5l2BDw7oA2AFzQJsZN"
    "gq+lm0lW9fBXXz+SWy80bdQtD3CegYYannMYLJcAYSASlHgYzSpeSEB43tMAgmQGbaFNClKxonfK/lVRZ7G8Bb17Yqp67/U944GN"
    "VxfoLAxRqlWNjrt6lr8GIyhNo+ImOLM6QXMQHq7qZWDXqF2UQZOZfDLq+0ZSqh0cAknofhPgXsB9HgzYwwF7dObpFy+iYk4uByGS"
    "FQlS7wn03QUOPtn9WF0gwI/PfJDg30swkHCGMPQpT2HuYXq+UsAeDUdn7AP2ycjXl/hBuKppWQEpqEUR8lrqe1DkIBQVfcKIA8ny"
    "2CXR8CVxYQUSNHDg/DW3VXAf+vh1PucZngcrOYVjbnjrJq1Z4pkGsT6Uc2aGZ/178WP1xDfB8ymPE+Rb7QZNYIIdEPktIzxVJs43"
    "DWsATqJScToXc128ZN3YLKwNmHjftnqf5qJaSww6tmwB9EX7MkRxpWfsPgJNEAkx8NGWPuIOSdWwMTdxFRI8OQM7dgx6hGAT4/MB"
    "oxeMQeLjzhzjnwFbwZLycQBzrRqnEIgY6N6DjmXfZYZgVfUaS4X7pqb1hwcEzbrZduCfs+8BY9J3zzoiG3pF3w0EAAFl3ULH4ADF"
    "Jxs7yzdoLKYUsG2comaFQFrrrRRm6EnPcDRyF9Oz9vbF1kiL0VUI0r0BBgUC3QzTqLxwH2SgtoXtp48dF8wFv8El6DlggBRbnf0H"
    "Z8g+8HUpgd4dCZ6S3eANxV6u6QXI2MSmoN+puE7FBoFRtPc09h2hchaBqgagiW7khnKxYRsOKBbR7fts+zDoHDJyNBrlZHdv78GZ"
    "MxYXoein8aKyC3TE/g17gCPYaMB465KHQp1BZcEBW1ZFL+pbcEhCPSYXvqAutREm0Zl4JyJdPEI+bj3RgO+aLyeug++OvpM37d41"
    "FxIkt6B3+J3HiZAqghid+Tf73VoSh/oXeV2R10fJHBzeJcozXFC0Eu+IXqx52TDp3SLOoCbIdYlhGAq6MzrzCmqIjaj1nY9K6bH1"
    "AjRW0OiAsezy7d0HAsZA3HQYno8N9LcanHLdML2HJxbnoL59HIlSadPoGiRZGa5AJgi8N/zBatXq1De8SX3WQAlBp5Za3QbbpVTt"
    "I9xDgIaPnWmReBCcR3N7YDuKt8KlkBydEysmBMIWBsTdNNspNdsBBa1vYoo8Hbv2cipcBjMMs+G9c8HSiCWd09qOxB4/d8GeRxWI"
    "w5sGXwJFuuHCABUcDS40ACaoZm+DCuj6zUV4WY9aKuE3cHWZgTQ0AGHShQkmY4DHgbALyFnmd4YIkG2+JlUH5966QRGE/FwJaHml"
    "xwS/w43G5QOL66OFrrrWoOlO+A0m3aEWNZQmZQTZ/g7xbunxAFMLiBV0VxmOJ50eSCsJnjM1XR+zpCjx+CgqK0nV8SIveaZIgJ4j"
    "EbA9aiTeLIFjmIwb/dcTvaU/BnUadNN1qCfNTWz8KQKKeBdyBDALtLePtjk34uCTkX1CVlfRHBsDP0nSGoSuJWOIbfO4yoXPBkfn"
    "Er/uLS+AQxAvpz6Tmp8Z3kkn0qMt16+uXglXPflrIGGZpRfzUUdXbT227bGts7E/dkUt41ivZ9sviOs3FqvceogLOsY/7Uc1H/s8"
    "2imPMrNkYxPzPpzho94lYclYZIhjjyP+NS9yAYY8e2Pk6pcugu+FgRLDglMVPLqQ4BTu2xPm0RTNXrEmYPVmJWyn6ZjeMRmdOUNo"
    "d0d32rK7//buOgBNUxR5BPCTshX5R8Y5WC4gJNIcjXu8yvIQXQOZ9BnJW9ItJn85agveq7OLLL/KxiM70hf3WH5ldtmeY1+IfW/t"
    "V/1U2B1Ct2pbGBjpDI+tgGfkPA2lIZmCgSXkGwaGLrnH0HL1uoGrzPXb8RiiJdpXI9JwoCH+2GXihfgCHFR3+IMr/vJSTxIPRi0j"
    "aQLQWkfU2J5OCoi9djtW2q9HEwcMArnYroOeDj0R9PfkatimZ3NInaAF8XggazsDFYkpOpkAJlALJ53p0XAkVKNPLFbrtRo739wg"
    "Vt8QSHEmR4Zy7VraGR3jkZmRTPsDNqJxjda9TgLpmGxbQ10Hy9lGTYi+k1rZVe62YKtJZbjLyZhkH46l99TSZlULrduTxijlqUIS"
    "T1fVTWgfg9SZjjUjEdsUnmR1wiLLrWEpmrYs1sqmI9eH9QqdEoYltUXQGtHjFTkeUSPmJOankDBWF6aZEDmk6iAvbIkc1zYFhJZk"
    "79074ybCLJ+X+261p9uv1m978N5vs15iraSUX2Qimbt22woEEGDtasGz0GrSHhs5Cq+iYmrrVTZQ2wyw7jvnfo1o62oBtu58saqr"
    "DQbQWpcNtcC+d7geP+7m45aNBOrO6/iCA5LbTo2sv9ehgpllEb37HZqRXJ4NV0PCIgchXXZFuDdw71e9/EEY3wjvYjQbRKfqkXtj"
    "WjwzaftcM6PJf9R+KvECTSZ+Fie24EYsDjdqZh1i3NmHZ9KWkgxwILV0aVSJPLMKyawHqh+sgTKkrHMyKxdONmI7rEc5RB8yETge"
    "9GWPLYk8/GHkCYaH+2PGL2y1jYIfyEsqf1m+GRV47/rHb9sia48SXIfTOl2VvUtQs/KiorQoYaiSb8Ac0UMDGpE8ppcBuZdbvuCo"
    "i4HU271RUHetJdYH6B73uE7w82EUbiNCRYoAIa/fPPZf8ikoVz3xj3uML+7h+Z+Kvq24OrVSzzA+P01eIymQQJSPhTpd5VW0JO9o"
    "kAKIYI+NpLsDdFD8JabKr3lcS//e2obay6nejg4bEJGO9qjcQJMAU3Pw1CxTA9FeoAgPtsRL9aTEbaFs6rseqDWsdoExUaINzBs3"
    "wjkG+cjz3RYUtu17o5EZNLqJHjEpQWvGogC1e3hfoQ7HxFsEgmm6Dx6NRsIdrJ4pjAsD7YF8igxEtLAUMOz/GFrYrEE0GugBtkJM"
    "uCalVZFfglaEO0eEAa0GMgkTlM8BM+36Oh/vkF+ZaI0EVhM6rCJyzWIMfZGkPANAFDiagNEIiiajLE4Z4E9xOJiHpwyTSXDJC2Ez"
    "IHtsJbLutf3hcoCkfQd0/rTKk6wKRTAZej8XsPvhF56NzgpeLj4aXQTEV71PmjH/KA62fG+ceN5G1AbS6iqM6ooO42HlCXQ4Cqs8"
    "1C8o+Kwkq5XSLK3A+kDuP0D+eQ1birIzY56sKifVyKQVEKQdlrlWKwmoxmm0GIdZmKEnJUQ2Eq+WnMjbTpODWMVQDrLZtm+HeugE"
    "MWBSNr5cBIWXu4FOGKDGpqyCu3/ooROfKAAaUm09bnW3SECk3lo8YjyWa3Bvs1ELLjD4LINGSAs+bq0A39eOOIIDXvW8WmB0gcDd"
    "tC6EJv6+HVs898frehpGW2ezZV0uVMcGy/2xFyP5MokpqiQqUnUiHjzbPsY9ikeMZ2u6GNRwriNGHj/eHbDHjx+s6ygS5FqIUQzI"
    "eQfySrvlR0084OmbwgVYl0o/w9si3dohdDz2KLrA2nqrDdDwcxuU3F1AzjLNxhv0sdXtkgXmXuVxvhybvTFoHhqkCRoYqKnsefgL"
    "KjA7qAFgO6kYehLO7zr2PobAoYpnH4yqSUVFlcyiGJUGlAKUOh3pAFSRXAU6XVHJ9CoCFi6xgkoY9NWxAkzxu3oBAQtn9XKp4dvH"
    "ZR6wRAcPdSQojdAEECA4carv0Ac2Qv9tYDMvS77ozH8xYR8PE2nmCJ8A2TPqAkatJRQqPFAaV0wqE8pUpzSf8mXZPMIlssUAV9VQ"
    "FDAIzs7agrJVwKEn3knoWJk3UJzEldQOQM0C02oqgmHMeAJbsxHbRuozskiDVFyUtfOBCbXEM7Elr7hK4G+kB5mg+lKGYaqQbQOv"
    "I6l/IO6KNh3FEigDgRr4SiZguJIqB8HSugR+y7F5mWB6W5rWZMPppC6BhUaBCQFIHD3hrOcgr4obS4Hr0h6EAuLVHgLFNKRQZ8ii"
    "QgVb37S0IKONCf5gMQC3REhv1WBvAuZEv3JdbJvsa8Lax+2SEdZcaQmbdRy6118XdwDiVCgPZYqMNR8TR+4mcvab8sLMDbr8ig7l"
    "Eq44RqO2h1NgY+LF7DrE6Eb9DoVVeRVI6zref/r8l0LTwnF6dr1XuBkzwcl8VpqsNWbT0iYHxfyym178vpxPZLqJTHAFG02kdAXb"
    "wAoHMkLOO0ZfGqtL685QG1i2DCHJw1pAvKza1krlMY7069+rDiu+pyAI3gdz+3Z2nLbghVtIFhZB7FqcvxUVg5rMQPl0dT0SJC6M"
    "lxzo34bx7z4eKSejZTOnbOwTNRMcTMvKTxV8IhS4IrqVQSXxMi+5q++kne/XQwemcF4CkpdjOvfsjrlNRYqcs/NSYlUSz5oLmyY0"
    "OozdxrPWUGTdtX0XNhUrGICBVBygkIwQEl290xlHExpFLAgkrZfSjTDt0YBUUtClH8I/ll8OhbEKDPDAIffVeBml59OIFXsMlV0L"
    "zdt6OO0ocITsXVHxQEOR076PlIRSes9eD7q02ObgxMNJcE4r2h6j89wZamP0TcAmR+o9RidRIO51rn2DRxEFkSJCom2WzCU3J0j4"
    "uw1JjyVEzzxuzrXO+mYMTTOxamazWVOkytQiIb6J3u6+D2WKNbm1q9ZsUpsVuFOxdjxsK73jRy4DGEn/tion4CMm6UxQbTyyop0f"
    "6ZHTrdRbJUg4xkD5PBYGuTIRP/CVXTAJ+Q0xuHZSa4o6rJmoIYbWW8zcdZs2Djy9FDLsXve5cvp+9GyZFIwNS1YM2vOwShoEnVUu"
    "rPgUN7+6e4JtkYqHKnzpmKBlEe90hGy2AE6g85nJAseAvjjBNrMkVtms0qQSgxZ+PS81eDx/SoEn1/fAPhvoOLLQB0UN46vlvTW6"
    "saOZqvZCOZUvFuqyerWv8Bi60cfk3aHkSHSbj3eVP9zLNFoZO9HakVAIyj3HDe30DP/pSGNQIlESRizyGVdACCyfUSLjLLnmU5zH"
    "QFa2i6ZTimaJlqxM0npJgpFIboE+eGEKF0Mdr0VQvbq6fGgv8ybLpvp1Ict+cyiznq3UPdAaBjpd2cCS/q/W5pAAWsnWlGNL+cLk"
    "PjAp1/rKrLfwh8g+pGkGvnQWhO5WYME4VOcOQvIENurIkt2HpEo1w2lA4o47lV2PsHYBqBRfMX75q9lIpv6ChjTwULy9HCKsZ2zO"
    "L9XDZqKlNsMsMrqdYVzv9m1UpHfbtwtZdRJTRchnQN7cQ+kaXNA9HBFqmP079hfWqe5Mul63bzNVutK4EbfMsesmFIlGqj1D17hW"
    "o98YHDrt2luWgD76WDv5bvGwj99Idbtjn9/JXq2EjE06UziUPjXFPUZHhXddroNvD18dN951I1AzNgHmvsNfr85gSvFShzUnsZYu"
    "oBsLXfie1j2gOJqf6tXEoggHAOLsrwn2aGnuVpkEDVk6hNHQccYp77sai3e3WBQ4YOds7GxXOtq0dx3MDGMI1raBeQVuMpewr/Za"
    "Aup6wChvDeNY8e2eCEm3Rm8PewwaTKPvDauYJXw5FVPFdAaVaqCz4uRNK0Wh31VTRr56QiCB39/oK3cgAA7zDqA1Kq5YHIjaifFd"
    "AQ1QaoGIlIG+mKmJYaTRZUkyRHjoYHpVknJY+0QWWxp4/pc8UqbjonUBfAjenOAo1GgwW430NGKrA/0/sqy+KxNBZStoKUo6YOx0"
    "HSoZ6WqQ1NsSyd+zRbJ3P1KCeYMXYi3OJFO/LIywWxjjXTtO1IlGNokaMFtor7L5dyxaVKKSyRT7mGd2oT111m3UFWvLyTlaG25r"
    "k53a7kbszbnbaVo3NAJ1+OP1VK9xZG816+FpSIJsYtAf5bHe7pr8b9VJKjEUHobKjQNRobijSAH1fR/vmwYs/Vl2VYPmM6uoAQG9"
    "1wmmAViuY+8LrOftl6z1jJkxek7Y9cO1gVPew1hhFHoHK59ZA11n36/Rc63pb6Tqan7Tiod2LWVV4zA21W8sf3x/2Dqlsgojt2RI"
    "7By3qpF6yNhX28ao1J4AbaU8y2Vue7UMCxTpDMxaBHnL9RSSHHFTIlQihF0joWUH0AFrR06D21qyfjdgRr7EykVodNIJd4Pu7a9H"
    "CnhIpjWAA8ElAupUmnFnF51L2S5F8HA0JF0pEXVP/O8Q6tIIm+q6VK2X2u7Eko4obi/2NMiLMxMC2rPwBzJbjY5KN1jvhd9OAke/"
    "nRKt5ImsrWFJYcqtMT8nqqncqqUVAtzRrNVq5qkh5Xd1SiCWx9NRnzcoYCOqcyXoqDH7QRdy8hfmgtYSD1oxYVTyU/6YYBOdoNkq"
    "JeKr0hVgF6wqqgG2i2i0NefWG/0vbOYmYfbuj1Arv1cLLZA3zrQXGfN+dU6rN+jbKB4gYslM9+ua+LUAVwOfWLYqrq58m7d3e5QA"
    "r3tc37mmbHO5QuvIMAZ9rfVkMadQjLipKmuJofVgb/Zndwbo+izQDTNBVR7DWJO9v43KYbBSFvwNWxkJ75Fr0FmdG8TkWKfV3B/T"
    "74djMnY2T9DxQ7LyjjbPLPKDwkT5ULQZe4rG3FcM3GV5Vto2DkR+eMYlY3GchHl9wj74xnnhW27pWrL7FAULcI3y9yLDWdbna6a9"
    "Yw1tGzp5GHaFW2vPl3IrraZ1b+zUR5q4MEq/QIOTu6/MF/UMmtsnXPp6zUph6kgtkjhk7aImovU+bu9ZCwzZ0gqDscXs0LrWmC37"
    "7+WHM7gUnrSR9r8ZztTXcZ3SmX7MZzVoK3VG1TC5LCZv1TDED7etIsqpGsgahBjDjn531CHzjNNeiZi2hMX3g4RXXX8ewPJmbqYw"
    "0zYgNq+c8I5pD2r3fJmf94IPPFoDFpduFErXHtLueKa9LfczAl6zYPZ+6v5GMwhaO2jWrBnajryknLUwaKadq/MaifexsXIG8hsH"
    "O3Wm3YL26goVRQbfpRdYh1mEBIucITzOAos9zC+sWhdC34HNM5+DQU/FaW2PD3FSLiJleioG2eICm7I6oaI6lbw1eKWq2IffnrK4"
    "WDQRpu9nH90FjK/WFoqmfC3JMTsbwnJe0ZmNsCdmGMhNwdm3+MfzkQ3b9dG7sn0UA4RECJPXiDXpbu9twNw9KbD2f/7MxDaI7F8U"
    "Mg//dSMTubv9BaDMX3t4bVkD8e2BJZXttUOL2DbLukJG9L7qVFbXK6yiNDefjvHPmrram2mu99Yy2aCmiVHqlhHptybEqrNtlosM"
    "9XG2UXNC8pj+rnk/HjM778eIJfsn1i2wf6/JC/YTTb/FHvlUqQ20qO1vDOFdD19MQTdMvd9mEhGMjcCma1mxQJ6WUVU98cLBxgAU"
    "zjeB4fngkBI1/2fp9rsixixUhC8SzNchSxMvBWONi/R+uhUts/Q+ki1S5AXpPUQb5jMCWGL7HSQMpNT0PvqUdoMwEa/R9QevxOQm"
    "qpMz2oxShAXNBZDovLQA9VXJnfcAtcxBEWqN6Eds+z3gdO07Ewgs8g2cBMKeq+XZR2XWa3qBlHTBQMk856mwsQJ1/OY8E9ZDoI7k"
    "nGdoPwTikM6FRywC4dFF611UqNnS45wGxnwI7NM+VZ16r5HP7Uukr0xNivySF6CEuxW/9DmvleX2DRTFeZHXq1Z+PNG399S1WKdz"
    "oGSWuoatecNQhVKO3hCtPxSu/tCpNbTceOiDpGFvWMIIqwTImC4YyZiqgtvMymJKwHzcbMDxA4+Fot1pbXemklDt7RE0vEO+JlQy"
    "49d5SYTc8Vi4q8GYko6Wjra/whgyXZysExoCobytZgVlT4dGo/Xz8X1mBfmkl9S8ck85GA3ZCVq1iUx/Fg2rlVBB9X63G7RtV211"
    "KMyYQLLuHY0hteG0FUeqWtWkLlTJyd1AqOl7O0kkeEKm7+uENSHCMkIfQ1hSeIoNo6ym/FLCwDmbYaAUMgLMrcfHs5wKxlNJTOnf"
    "FHQhi0RhjTBZlWktlggfdsGjWSGLWa7yfCmKz/s+LWe/0qkB1XwbSGN7tDBD+6d/eo3PcAq2q7QmXb3Nx7MD2Rg/5iWujODToRdG"
    "ru15ohpk3IKKHKhF1VAZeKFtMFPYBA+ry0alKyuwxNom7kF+1xG+BRow2mvWgRTFRmZSlobisCI0R1GN/T95D0/+Mjrny/EsePX0"
    "i9vojh19+unJ/ultxD5gD+86PkAU58u8GNPfLucyomc86RnnPiOIolK05Xum2rfWlhY9zzbRrGXhXxqGe9rWC74/Gv3wwfkDFM3f"
    "nz5+zEcjuhyNPuE/fEiXcfzDT6If0iX/6JMZBl41JN+gcVpSLflYfZWbfDf7Bv/sxcGn+89++ezFfgNlNwK9iF2JWkDByenT01cn"
    "bPSU7R6zB8+a58XZlD750giBLi4wh3XSGNa1hL//xdMXr56eHhwdspMXR6esN2KnR+jDbgCf5XkFIpyPg5dHJ6fbJ6f7L+VwnrAR"
    "e/rs9OCLfWBAx/unr44PDw4/Yw/Ys58+Pf4MLp+wo8N99uzo8PTg8NURjP+zp5/vtypCbOZD00diWmR7RDQLBFsTuajiDEUcI8tT"
    "kOZJmRXV1Y7WRsVD00pPa26KUvDTs6TGWSTT73s/I6m+ybzZ3geSwbg92+tz5j3ElIC7Pl+29qBuMzvS3u40KHayv/9cDmqwttu9"
    "W7659dc2Ij0dP4Vwv9qh8S3QKj91hJyz37/3HeZoTH0D6aPR7vpua7wpm3xAjxYSl8pzyt1i7B4Gf0vzd6hlsIYgOo5CJa+y9iry"
    "B4ybvXfBJcOiYQwpAaLXYW56+VSLX+EyB8L4CawC4yJ2/CHmPjx6POp3YP19uVuby32+//QQGC+Y0AeHJy/3n53uPzd+dzDKTn96"
    "cEh8jeoRIcNDDD1hh0fs5POjI3z8WbBOGjX4mWNEtAyB/wf8ymgqZx2BvD4zdbK79/isI4jiKus0I0i3lFt4/ScrN3Se38NI17vP"
    "v4m9221DyBHB9LvDMwSyFZfuideAwBcnyzv4DaQWywOAPruJSu4itE3kAbF9sVcM3UhurTiy+Kf/L5hdCdpjn++fHh88O/m/z6re"
    "lwnJ8QI3Od0/PrmfFwHL26aecpM9YT97StyHvdw/3n61DywKAMFbGHGt46en+yfEiY73QW88WcuGfKzkvbZ317YutDey2yMlnHQd"
    "Dinrm2qoFn53nijvTnT3xTfagR17wr8frL0gsBASttZviIm1X5Gvs89BSp8wwc3F3rUsGLl/pRPTQ6Rq/7x8enAM4u3V6cGLg9Nf"
    "rt8/yi55vv/i9Knq4mnXuXPW7ZpvsWMs+0TM5gSMDC2xn/hF9SnsnOf7L/fhz+GpeXy8//LFwTN6f3PzWDXvsZ4JRiGgRn+e58te"
    "M3jFhKU0A1b6pmAPBp+IYjyuCo5lu1dYl4ZAN16oUk9N2GmdqW/49p0PNVIvCuwuZeexC2vQ+iq79TlhzJrgpfk8cNc77SLQ4h7V"
    "nmtDLMdUZcIHw5Q5s6pzizCzHimksETh0eGLXwbOF2wtCMI9BOt/cuLGyDQXxq2xHBwcPjv6/OWL/VPbKLUPN8aNRRw4ofWUODBu"
    "hB8NGp9BkIlpY5Uz3aoTKp+rnwP3A2piK4f4cVtYBlGxqp1uL0BYe66Vv2A/lGGp+P0JETo2bkapq4BXq0kjs8pK4clXsPcwPG+O"
    "aVYMVwFP+HX5TywCbMLJADY63Ob8CcvGD+BPDqQDPCfbgTfAkzmGTuOnAJM0cD4O6qtOHNg7SVWhsu+1+261+HCgSSrUs2jWrmxb"
    "yWuqhq0pKOUK0O9i3333e6/vlpWWZbWMnA++T1/HjfXXfNFK/F9/+Z/Y+duvchbDHwtxgXU9C06Ld19/lc1ZtXj7VbLHbu2lmvxA"
    "bPofnN09YbeNjXe382jk7MwnmvhHQ/t1X+jyMOxPX779Hbvm6ROVM43jzN59/Q8py+b5279NNAhQs4gaTQFbcvcP2al2MI0ffjR6"
    "+xtdPORi8fZ/Yozg4t3Xv0vYIkrYOQCmj1a/e/MH+PdPX7578zcxW777+o8rZ4DH5IraeUbsYKfgv8rRF/TuzT+ILtl8yF6VfFYv"
    "mTwwIpD/nlVv/5gtsOHfiY/ZllHNRO8hO1y8/ceUHdMH54p3b34D71++/VsY0psvExbDEKHnr+soY+Xbr0BTihf//PeRHmmawNNs"
    "Xt+8/W8ZyxZvf5sNPet3pqILfVRkJegLcvmwqRfOgmfvvv7t4U/ZT97+1dGeXt62DLtjmmzZHMYfK1THb/9H9oRw9NcxIvt3GZsn"
    "hP3s7R+yHXyAk13AvX+qVaULXbSE/VxAuX77VSxm/Kcv//nv3735Xcxwy9Fn49mfHbzc+Qz+iBc/MZJYFgoEgvrtDWF2hWjCJYkX"
    "zJZQTcJwueKwWTj358m7N/8uRZQy8XVP5Jc76gvVpuqvYNhM5LGySxiAYaeX777+O0LFm/+uVnHIni2AuBETFVEMDAgDJPdYF6dr"
    "jcz6fWZCdBtFx+5f9uAXHCtzloJul4RxWf2Mpe/e/OfELEO8yBGtf1g9aeDQiEHYTW/+A2C8fPfmj80RI2+Re7dIAAfzPSYL4Y5v"
    "WwOf/KCzYC5wH1PWhL5iBut/HwRXlBMIXRikurGLgnhBbVRKl9giJg3qDYGsBTYs/FOxK4Hn+9dRW2A6EKK5il6P8SzYZidooBTA"
    "puFfGI34pgTeIIsJ7mzjDzSb4HoPbJPjMd4wUTw/ONsbfjQD5IDZQo+cGB/9VMTz4HO6gvsf4oMn7Bc7pzsv6AHG4sBLdvAaQ2rU"
    "tYiJgV/DwHPIcE8IhrRna+s7jNElOefrdUEStc4tDOkkU9YVkPe2sXKsU1lrLY7JX7XH8OznPEIR9u7r/1qLf27YKpFCrQQpGhMf"
    "gD3y5vdwCX9R1pBQxTPXGqiKpEgoz/+UBgwrx34s81JojmjKAvKQIdOlERQ4im0jg5v7Enbhl0ZgwryWUpCXXuQT0cmzVidXvcu0"
    "Jx8mfsn78Qg9l7uPZYoFXD9S1w3v5RrkagQDlQCON6Dff5ufEK1Z/lkk0YezO/Zq/wn7GVYYwueuwxabPEJidaSxOCOn5mvO0F26"
    "belgDne1d3bwTCjWlFsNi4G+EDD0YiEq0DbZjqrtIikv9oTknMvAqh1KxmBH4mOB2Dhe1DBiZMtv/zEhcbSnsmHYK6H87JBSXy14"
    "LkgAadX6jM2eSbDxfLRmyL4AHg5k8/aPUnvZEf4WQa00BgQoVJQK0PdfBF07lI7C/q9qIztI+jlay+egJiTmmL4Ckv19hZMHSEJW"
    "gjFDSR7ogdmxEtvYi5alwhaotaLC9PU/qU2AQ/iP8G82X/zpDxFKpt9Y4l7rG6hk/XUmp2BJMkQfELDeUxckq5c1MPNM2kMMLCHc"
    "LcnrSFTrjOuiSOJ6Wac/nsKo8WMmICWi6c02JQGCInMDI33zNwlK298PHcXNMpmO918eHZ+GXxwM02nQH8roCAx2D/48C4a4Jj1B"
    "cX2MSPlzNKmAmHJMihkHdTXb/jhwPjljK/GyBjMWVVF1m0BrKanAGCi8dD18Wsxr/OjAS3rSm/IyLpIVdh+H4TSPw7Bv9RxG02kY"
    "yS494KswD4zSuwGzE8str2m7vS2mbTfXdaCnTpaHt7dsua1LZwyY2LDjAD8BgJ91r3mwFgTZgtvaFtw2Rtk6YAABPbMSJv2DUFVt"
    "n1WB1TI0tVsfwmlUlhCVsNsRxgBqSIgc+J/JMJm2a7dZQptat+qGtIGuMZYRwqYWc8NZjxldWTV+0K4zv7UFkiUMMfIiDEkehyES"
    "ZRjK8ieCQrf+N6oPl7A=")

reader = zlib.decompress(base64.b64decode(ANALYZER_PAYLOAD))
assert hashlib.sha256(reader).hexdigest() == "700a3e23f1f647ced1d02501c60e2a5526a5fa5d0ebd0edc904ec354ff2965d3"
compile(reader, "analyze_bs5_continuous.py", "exec")
(TOOLS / "analyze_bs5_continuous.py").write_bytes(reader)
print("Transport-aware reader ready; missing original container hashes will be reported explicitly.")


In [ ]:
OUT = WORK / "bs5_dataset_recovery"
index = 1
while OUT.exists() and not (OUT / "recovery_receipt.json").is_file():
    OUT = WORK / f"bs5_dataset_recovery_{index}"
    index += 1
print("Output:", OUT, flush=True)
# auto preserves an audited evaluation if training exports are missing, and
# includes only selected weights whose bytes match the recorded SHA256.
command = [
    sys.executable, "-B", "-u", str(TOOL), "--delivery-mode", "auto",
    "--allow-extracted-containers",
    "--workflow", str(FLOW),
]
if not (EVIDENCE / "source_snapshot.zip").is_file():
    command += ["--eval-source-snapshot", str(SOURCE_SNAPSHOT)]
command += ["--output", str(OUT)]
subprocess.run(command, check=True, cwd=WORK,
               env=dict(os.environ, PYTHONDONTWRITEBYTECODE="1"))
receipt = json.loads((OUT / "recovery_receipt.json").read_text())
assert receipt["content_checks_passed"] and receipt["added_training_steps"] == receipt["added_environment_steps"] == 0
print("Analysis status:", receipt["integrity_status"], "; original-container integrity accepted:", receipt["accepted"])
print("Original evidence unchanged:", receipt["original_evidence_unchanged"])
print("Best weights included:", receipt["selected_weights_count"], "/ 4")


In [ ]:
availability = json.loads((OUT / "training_availability.json").read_text())
print("Download the results ZIP and receipt below; keep the ZIP before ending the session.")
for path in (Path(receipt["path"]), OUT / "recovery_receipt.json", OUT / "training_availability.json"):
    print(path, f"({path.stat().st_size / 1024**2:.2f} MiB)")
    display(FileLink(path.relative_to(WORK).as_posix()))
display(Markdown((OUT / "analysis/REPORT_VI.md").read_text(encoding="utf-8")))
if not availability["complete_training_export"]:
    print("Original training export incomplete; evaluation remains independently audited.")
    print("Missing training file(s):", availability["missing_training_files"])
if not availability["includes_selected_weights"]:
    print("No checkpoint substitution or training was performed. Exact missing weights:")
    for item in availability["missing_selected_weights"]:
        print(item["model"], item["relative_path"], "SHA256:", item["expected_sha256"])
if not receipt["accepted"]:
    print("CONTENT_ONLY: extracted-data checks passed; original ZIP/GZIP hashes cannot be verified.")
    display(FileLink((OUT / "analysis/transport_integrity.json").relative_to(WORK).as_posix()))
